In [2]:
import argparse
import json
import os
import re
from pathlib import Path

import pandas as pd

In [3]:
# -----------------------------
# Helpers
# -----------------------------
def clean_text(x):
    if pd.isna(x):
        return ""
    s = str(x)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def is_id_like(col: str) -> bool:
    cl = str(col).lower()
    return any(k in cl for k in ["id", "index", "uuid", "key"])


def normalize_label(x: str) -> str:
    # Normalize labels like "c1" / " C1 " -> "C1"
    s = clean_text(x).upper()
    s = re.sub(r"\s+", "", s)
    return s


def extract_label(text: str, allowed_labels: list[str]) -> str | None:
    """Best-effort extraction if a model outputs extra words."""
    if text is None:
        return None
    t = normalize_label(text)
    # exact match first
    if t in allowed_labels:
        return t
    # search inside text (e.g. "The answer is C3")
    for lab in allowed_labels:
        if lab in t:
            return lab
    return None


def guess_column(df, candidates):
    """Return first candidate that exists in df (case-insensitive)."""
    cols = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols:
            return cols[cand.lower()]
    return None


def find_best_text_column(df):
    """
    Heuristic: pick the column with the highest average text length
    among object/string columns, excluding obvious ID columns.
    """
    obj_cols = [c for c in df.columns if df[c].dtype == "object"]
    if not obj_cols:
        return None

    scores = []
    for c in obj_cols:
        if is_id_like(c):
            continue
        lens = df[c].astype(str).map(len)
        scores.append((lens.mean(), c))
    if not scores:
        return obj_cols[0]
    scores.sort(reverse=True)
    return scores[0][1]


def find_best_label_column(df):
    """
    Heuristic: common label/answer columns; else choose a short text column
    with low unique ratio (often categorical labels).
    """
    common = [
        "answer", "label", "target", "output", "expected", "truth",
        "root_cause", "cause", "fault", "resolution", "action", "category"
    ]
    col = guess_column(df, common)
    if col:
        return col

    # If no obvious label, guess a column that is not ID and has fewer unique values
    candidates = []
    for c in df.columns:
        if df[c].dtype == "object":
            if is_id_like(c):
                continue
            nunq = df[c].nunique(dropna=True)
            candidates.append((nunq, c))
    if not candidates:
        return None
    candidates.sort()
    # pick the one with smallest unique count (more label-like)
    return candidates[0][1]


def build_chat_example(user_text: str, assistant_text: str, allowed_labels: list[str]):
    allowed_str = ", ".join(allowed_labels)
    system_msg = (
        "You are a telecom troubleshooting classifier. "
        f"Your job is to output ONLY one label from: [{allowed_str}]. "
        "Do not output explanations."
    )
    user_msg = (
        user_text +
        "\n\n"
        f"Output format: Return ONLY one label from [{allowed_str}] (example: {allowed_labels[0]})."
    )
    return {
        "messages": [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": assistant_text},
        ]
    }


def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def stratified_split(df: pd.DataFrame, label_col: str, val_ratio: float, seed: int):
    """Stratified split to keep label distribution stable."""
    import random
    rng = random.Random(seed)
    val_idx = []
    for lab, g in df.groupby(label_col):
        idx = list(g.index)
        rng.shuffle(idx)
        n_val = max(1, int(round(len(idx) * val_ratio)))
        val_idx.extend(idx[:n_val])
    val_df = df.loc[val_idx].copy()
    tr_df = df.drop(index=val_idx).copy()
    return tr_df.reset_index(drop=True), val_df.reset_index(drop=True)


def print_basic_eda(train_df: pd.DataFrame, input_col: str, label_col: str):
    print("\n=== Basic EDA ===")
    print("Rows:", len(train_df))
    print("Missing input:", int(train_df[input_col].isna().sum()))
    print("Missing label:", int(train_df[label_col].isna().sum()))

    labels = train_df[label_col].map(normalize_label)
    vc = labels.value_counts().sort_index()
    print("\nLabel distribution:")
    print(vc.to_string())
    print("\nLabel %:")
    print((vc / vc.sum() * 100).round(2).to_string())

    # Text length stats
    lens = train_df[input_col].fillna("").astype(str).map(len)
    print("\nInput length (chars) stats:")
    print(lens.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_string())

    # Duplicates on input text
    dup_count = int(train_df[input_col].astype(str).duplicated().sum())
    print("\nDuplicate input rows:", dup_count)
    if dup_count > 0:
        dups = train_df[train_df[input_col].astype(str).duplicated(keep=False)][[input_col, label_col]].head(10)
        print("Examples of duplicates (first 10):")
        print(dups.to_string(index=False))


def validate_jsonl_rows(rows: list[dict], allowed_labels: list[str], n_show: int = 3):
    # Ensure assistant content is strictly one label
    bad = 0
    for r in rows:
        a = r["messages"][2]["content"]
        if normalize_label(a) not in allowed_labels:
            bad += 1
    print(f"\nJSONL validation: {len(rows) - bad}/{len(rows)} assistant labels OK")
    if bad:
        print("[!] Some assistant outputs are not in allowed labels. Fix label normalization.")

    print("\nSample records:")
    for i in range(min(n_show, len(rows))):
        print(json.dumps(rows[i], ensure_ascii=False, indent=2)[:900])
        print("---")


# -----------------------------
# Main
# -----------------------------
def main(data_dir: str, out_dir: str, val_ratio: float, seed: int):
    data_dir = Path(data_dir)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    train_path = data_dir / "train.csv"
    test_path = data_dir / "phase_1_test.csv"
    sub_path = data_dir / "SampleSubmission.csv"
    truth_path = data_dir / "phase_1_test_truth.csv"

    print("\n=== Loading files ===")
    if not train_path.exists():
        raise FileNotFoundError(f"Missing: {train_path}")
    train_df = pd.read_csv(train_path)
    print(f"Loaded train.csv: {train_df.shape}")

    if test_path.exists():
        test_df = pd.read_csv(test_path)
        print(f"Loaded phase_1_test.csv: {test_df.shape}")
    else:
        test_df = None
        print("phase_1_test.csv not found (ok).")

    if sub_path.exists():
        sub_df = pd.read_csv(sub_path)
        print(f"Loaded SampleSubmission.csv: {sub_df.shape}")
    else:
        sub_df = None
        print("SampleSubmission.csv not found (ok).")

    if truth_path.exists():
        truth_df = pd.read_csv(truth_path)
        print(f"Loaded phase_1_test_truth.csv: {truth_df.shape}")
    else:
        truth_df = None
        print("phase_1_test_truth.csv not found (ok).")

    # -----------------------------
    # Inspect columns
    # -----------------------------
    print("\n=== Train columns ===")
    print(list(train_df.columns))
    print("\n=== Train head(3) ===")
    print(train_df.head(3).to_string(index=False))

    # -----------------------------
    # Guess schema (input/question vs label/answer)
    # -----------------------------
    input_col = guess_column(train_df, ["prompt", "question", "query", "input", "text", "problem", "log", "context"])
    if input_col is None:
        input_col = find_best_text_column(train_df)

    label_col = guess_column(train_df, ["answer", "label", "target", "output", "truth"])
    if label_col is None:
        label_col = find_best_label_column(train_df)

    print("\n=== Auto-detected mapping (best effort) ===")
    print(f"INPUT column  => {input_col}")
    print(f"LABEL column  => {label_col}")

    if input_col is None or label_col is None or input_col == label_col:
        print("\n[!] Could not reliably detect input/label columns.")
        print("Open train.csv and decide which columns are the question/log (input) and the expected answer (label).")
        return

    # Normalize label column for stability
    train_df[label_col] = train_df[label_col].map(normalize_label)
    allowed_labels = sorted(train_df[label_col].dropna().unique().tolist())
    print("\nAllowed labels:", allowed_labels)

    # EDA summary
    print_basic_eda(train_df, input_col=input_col, label_col=label_col)

    # -----------------------------
    # Build user prompt (label-only classification)
    # -----------------------------
    context_cols = [c for c in train_df.columns if c != label_col and not is_id_like(c)]
    if input_col in context_cols:
        # keep input_col last for readability
        context_cols = [c for c in context_cols if c != input_col] + [input_col]

    def make_user_text(row):
        parts = []
        for c in context_cols:
            val = clean_text(row.get(c, ""))
            if not val:
                continue
            parts.append(f"{c}: {val}")
        return "\n".join(parts)

    # -----------------------------
    # Create dataset in chat jsonl
    # -----------------------------
    print("\n=== Building JSONL (chat format) ===")
    df = train_df.sample(frac=1.0, random_state=seed).reset_index(drop=True)  # shuffle
    tr_df, val_df = stratified_split(df, label_col=label_col, val_ratio=val_ratio, seed=seed)
    print(f"Split (stratified) => train: {len(tr_df)} | val: {len(val_df)}")

    train_rows = []
    for _, r in tr_df.iterrows():
        user_text = make_user_text(r)
        ans = normalize_label(r[label_col])
        train_rows.append(build_chat_example(user_text, ans, allowed_labels))

    val_rows = []
    for _, r in val_df.iterrows():
        user_text = make_user_text(r)
        ans = normalize_label(r[label_col])
        val_rows.append(build_chat_example(user_text, ans, allowed_labels))

    train_out = out_dir / "train.jsonl"
    val_out = out_dir / "val.jsonl"
    write_jsonl(train_out, train_rows)
    write_jsonl(val_out, val_rows)

    print(f"\n✅ Wrote: {train_out}")
    print(f"✅ Wrote: {val_out}")

    # Validate and show examples
    validate_jsonl_rows(train_rows, allowed_labels, n_show=2)

    # -----------------------------
    # Optional: inspect test/submission alignment
    # -----------------------------
    if test_df is not None and sub_df is not None:
        print("\n=== Submission sanity checks ===")
        print("Test columns:", list(test_df.columns))
        print("SampleSubmission columns:", list(sub_df.columns))
        common_cols = set(test_df.columns).intersection(set(sub_df.columns))
        print("Common columns between test & submission:", sorted(common_cols))

        # If truth exists, show label set alignment
        if truth_df is not None:
            # Guess label column in truth file
            truth_label_col = guess_column(truth_df, ["answer", "label", "target", "output", "truth"])
            if truth_label_col is not None:
                truth_labels = sorted(truth_df[truth_label_col].map(normalize_label).dropna().unique().tolist())
                print("Truth labels:", truth_labels)
                print("Train labels:", allowed_labels)


# For Jupyter Notebook: call main directly with your parameters
data_dir = "."  # folder containing train.csv
out_dir = "prepared_data"
val_ratio = 0.10
seed = 42

main(data_dir, out_dir, val_ratio, seed)


=== Loading files ===
Loaded train.csv: (2400, 3)
Loaded phase_1_test.csv: (864, 2)
Loaded SampleSubmission.csv: (6908, 4)
Loaded phase_1_test_truth.csv: (3456, 4)

=== Train columns ===
['ID', 'question', 'answer']

=== Train head(3) ===
           ID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [ ]:
import json
import numpy as np
from transformers import AutoTokenizer

# =================CONFIGURATION=================
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # Le modèle que vous ciblez
DATA_PATH = "train.jsonl"                # Votre fichier généré
# ===============================================

print(f"Chargement du tokenizer pour {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

lengths = []

print("Analyse des longueurs de tokens...")
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        entry = json.loads(line)
        # On applique le template de chat pour voir la longueur REELLE vue par le modèle
        # Cela inclut les balises <|im_start|>, system prompt, etc.
        text = tokenizer.apply_chat_template(entry["messages"], tokenize=False)
        tokens = tokenizer(text)["input_ids"]
        lengths.append(len(tokens))

# Conversion en numpy pour les stats
lengths = np.array(lengths)

print("\n=== RÉSULTATS DE L'ESTIMATION ===")
print(f"Nombre d'exemples : {len(lengths)}")
print(f"Longueur Min      : {np.min(lengths)}")
print(f"Longueur Moyenne  : {int(np.mean(lengths))}")
print(f"Longueur Max      : {np.max(lengths)}")
print(f"95ème percentile  : {int(np.percentile(lengths, 95))}")
print(f"99ème percentile  : {int(np.percentile(lengths, 99))}")

print("\n=== RECOMMANDATION ===")
p99 = int(np.percentile(lengths, 99))
# On arrondit à la puissance de 2 supérieure ou au standard le plus proche
possible_lengths = [2048, 4096, 8192, 16384, 32768]
recommended = next((x for x in possible_lengths if x >= p99), p99)

print(f"Si votre VRAM le permet, fixez max_seq_length = {recommended}")
print(f"Cela couvre 99% de vos données sans troncature.")

In [ ]:
import json, random
p="data_jsonl/train.jsonl"
lines=open(p,encoding="utf-8").read().splitlines()
for i in random.sample(range(len(lines)), 3):
    obj=json.loads(lines[i])
    print(obj["messages"][0]["content"])
    print(obj["messages"][-1]["content"])
    print("-----")


In [ ]:
import json, numpy as np
from transformers import AutoTokenizer
tok=AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
lens=[]
for line in open("data_jsonl/train.jsonl",encoding="utf-8"):
    obj=json.loads(line)
    text=tok.apply_chat_template(obj["messages"], tokenize=False)
    lens.append(len(tok(text)["input_ids"]))
lens=np.array(lens)
MAX_LEN=2048
print("Truncated %:", (lens>MAX_LEN).mean()*100)
print("P95:", np.percentile(lens,95))


In [ ]:
import json, numpy as np
from transformers import AutoTokenizer
tok=AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
lens=[]
for line in open("data_jsonl/train.jsonl",encoding="utf-8"):
    obj=json.loads(line)
    text=tok.apply_chat_template(obj["messages"], tokenize=False)
    lens.append(len(tok(text)["input_ids"]))
lens=np.array(lens)
MAX_LEN=3072
print("Truncated %:", (lens>MAX_LEN).mean()*100)
print("P95:", np.percentile(lens,95))

In [ ]:
import pandas as pd, json, re
from collections import Counter

re_lab=re.compile(r"C[1-8]")
def lab_from_jsonl(path):
    labs=[]
    for line in open(path,encoding="utf-8"):
        obj=json.loads(line)
        a=obj["messages"][-1]["content"]
        labs.append(re_lab.search(a).group(0))
    return Counter(labs)

print("train:", lab_from_jsonl("data_jsonl/train.jsonl"))
print("val  :", lab_from_jsonl("data_jsonl/val.jsonl"))
